In [3]:
import pandas as pd


In [4]:
imdb = pd.read_csv(
    './data/title.basics.tsv.gz',
    sep="\t",
    na_values = '\\N', 
    dtype=str,
    )

movies = imdb[imdb['titleType'] == 'movie'][
    ['tconst', 'primaryTitle']
]

In [5]:
directors = pd.read_csv(
    './data/title.crew.tsv.gz', sep='\t', na_values='\\N', dtype=str
)[['tconst', 'directors']]

movies = movies.merge(directors, on='tconst', how='left')

In [6]:
movies

,tconst,primaryTitle,directors
0,tt0000009,Miss Jerry,nm0085156
1,tt0000147,The Corbett-Fitzsimmons Fight,nm0714557
2,tt0000502,Bohemios,nm0063413
3,tt0000574,The Story of the Kelly Gang,nm0846879
4,tt0000591,The Prodigal Son,nm0141150
...,...,...,...
759758,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,"nm9272490,nm9272491"
759759,tt9916680,De la ilusión al desconcierto: cine colombiano...,nm0652213
759760,tt9916706,Dankyavar Danka,nm7764440
759761,tt9916730,6 Gunn,nm10538612


In [7]:
movies = movies.rename(columns={
    'tconst': 'imdb_id',
    'primaryTitle': 'title'
})

movies

,imdb_id,title,directors
0,tt0000009,Miss Jerry,nm0085156
1,tt0000147,The Corbett-Fitzsimmons Fight,nm0714557
2,tt0000502,Bohemios,nm0063413
3,tt0000574,The Story of the Kelly Gang,nm0846879
4,tt0000591,The Prodigal Son,nm0141150
...,...,...,...
759758,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,"nm9272490,nm9272491"
759759,tt9916680,De la ilusión al desconcierto: cine colombiano...,nm0652213
759760,tt9916706,Dankyavar Danka,nm7764440
759761,tt9916730,6 Gunn,nm10538612


In [8]:
names = pd.read_csv(
    './data/name.basics.tsv.gz',
    sep = '\t',
    na_values = '\\N',
    dtype =str )[['nconst', 'primaryName']]

d = movies [['imdb_id', 'directors']].dropna().copy()
d['nconst'] = d['directors'].str.split(',')

d = d.explode('nconst')

d = d.merge(names, on ='nconst', how = 'left')



d['primaryName'] = d['primaryName'].fillna('')
director_names = (
    d.groupby('imdb_id')['primaryName']
    .agg(','.join)
    .rename('director')
    .reset_index()
)

movies = movies.merge(director_names, on='imdb_id', how='left')
movies = movies.drop(columns='directors')

In [9]:
movies


,imdb_id,title,director
0,tt0000009,Miss Jerry,Alexander Black
1,tt0000147,The Corbett-Fitzsimmons Fight,Enoch J. Rector
2,tt0000502,Bohemios,Ricardo de Baños
3,tt0000574,The Story of the Kelly Gang,Charles Tait
4,tt0000591,The Prodigal Son,Michel Carré
...,...,...,...
759758,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,"Angela Gurgel,Ana Célia de Oliveira"
759759,tt9916680,De la ilusión al desconcierto: cine colombiano...,Luis Ospina
759760,tt9916706,Dankyavar Danka,Kanchan Nayak
759761,tt9916730,6 Gunn,Kiran Gawade


In [11]:
movies = movies.dropna(subset=['director'])
movies

,imdb_id,title,director
0,tt0000009,Miss Jerry,Alexander Black
1,tt0000147,The Corbett-Fitzsimmons Fight,Enoch J. Rector
2,tt0000502,Bohemios,Ricardo de Baños
3,tt0000574,The Story of the Kelly Gang,Charles Tait
4,tt0000591,The Prodigal Son,Michel Carré
...,...,...,...
759758,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,"Angela Gurgel,Ana Célia de Oliveira"
759759,tt9916680,De la ilusión al desconcierto: cine colombiano...,Luis Ospina
759760,tt9916706,Dankyavar Danka,Kanchan Nayak
759761,tt9916730,6 Gunn,Kiran Gawade


In [14]:
dmb_long = d[['imdb_id', 'nconst', 'primaryName']].rename(columns={'primaryName': 'director'})
dmb_long = dmb_long[dmb_long['director'] != '']          # cell 5 filled missing names with ''
dmb_long = dmb_long.merge(movies[['imdb_id', 'title']], on='imdb_id')   # movies already has no-director rows dropped
dmb_long = dmb_long.drop_duplicates(subset=['imdb_id', 'nconst'])
dmb_long.to_csv('dmb_movies.csv', index=False)